In [20]:
# 데이터 생성
import pandas as pd

dataset = pd.read_csv('../../data/csv/movie_reviews3.csv')
dataset.head()


,label,review
0,5.0,영화가 너무 재미있어서 하나도 안 지루해요.
1,0.5,영화가 하나도 안 재미있어서 너무 지루해요.
2,3.0,기대 안 했는데 생각보다 너무 괜찮네요.
3,0.5,기대 많이 했는데 생각보다 너무 별로네요.
4,3.0,배우 연기는 별로지만 스토리가 살렸습니다.


In [21]:
X = dataset.iloc[ : , -1]
# 0.5 -> 0으로, 3 -> 1로, 5 - >2로
def convert(score):
  return 0 if score == 0.5 else 1 if score == 3 else 2
y = dataset['label'].apply(convert)

In [22]:
# 형태소 분석
from konlpy.tag import Okt
okt = Okt()

def tokenized_korean(text_list):
  return [" ".join(okt.morphs(text, stem=True)) for text in text_list]

morphs_korean = tokenized_korean(X)

In [23]:
# 벡터화

from tensorflow.keras import layers, models

vectorize_layer = layers.TextVectorization(
  max_tokens = 1000,
  output_mode='int',
  output_sequence_length = 10	 # 단어 10개
)

vectorize_layer.adapt(morphs_korean)


In [24]:
# 파이프라인

import tensorflow as tf
train_ds = tf.data.Dataset.from_tensor_slices((morphs_korean, y)).batch(8)

In [25]:
# 모델 설계
def bulid_positional_encoding():
  
  # 입력층
  inputs = layers.Input((1,), dtype=tf.string) # 1은 입력받을 변수 (문장 하나씩 받으니까 1개)
  
	# 임베딩
  x = vectorize_layer(inputs)
  vocab_size = vectorize_layer.vocabulary_size()
  word_embedding = layers.Embedding(input_dim=vocab_size, output_dim=64)(x)
  
	# 포지셔널 인코딩
  # 위치 정보 추가 : 0~9 번호를 생성
  positions = tf.range(start=0, limit=10, delta=1)
  
	# 임베딩 : 위치를
  pos_embedding = layers.Embedding(input_dim=10, output_dim=64)(positions)
  
	# 의미 + 위치
  x = word_embedding + pos_embedding
  
	# 멀티 헤드 어텐션
  attention_output = layers.MultiHeadAttention(num_heads=2, key_dim=64)(x,x)
  
	# 잔차 계산 및 정규화
  x = layers.Add()([x, attention_output])
  x = layers.LayerNormalization()(x)
  
	# ffn
  ffn = layers.Dense(64, activation='relu')(x)
  
	# 잔차 연결 및 정규화
  x = layers.Add()([x, ffn])
  x = layers.LayerNormalization()(x)
  
	# 압축
  x = layers.GlobalAveragePooling1D()(x)
  
	# 출력층
  outputs = layers.Dense(3, activation='softmax')(x) 
  return models.Model(inputs=inputs, outputs=outputs)

model = bulid_positional_encoding()
  

In [26]:
# 모델 설정
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [27]:
print(set(y))

{0, 1, 2}


In [28]:
# 모델 학습
model.fit(train_ds, epochs=100, verbose=0)

In [59]:
import numpy as np

# 예측
# 테스트
test_text = ["영화가 너무 재미있어서 하나도 안 지루해요",
             "영화가 너무 지루해서 하나도 안 재미있어요",
             "배우 연기가 별로네요"
						 ]

# 형태소별로 문장을 수정
test_morphs = tokenized_korean(test_text)

# 테스트 할 때 텐서 타입을 변환(문자열 이슈)
test_input = tf.constant(test_morphs)

# 예측 실행
predictions = model.predict(test_input)
predictions_size = len(predictions)

print(predictions[0]) # 첫번째 리뷰의 결과값

labels = [0.5, 3, 5]

# numpy의 argmax를 이용

for i in range(predictions_size):
  idx = np.argmax(predictions[i])
  # print(idx)
  # print(predictions[i])
  p = predictions[i][idx]  * 100
  print(f'{test_text[i]} : 별점: {labels[idx]}점({p:.2f}%))')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
[2.7471997e-03 8.3901675e-04 9.9641377e-01]
영화가 너무 재미있어서 하나도 안 지루해요 : 별점: 5점(99.64%))
영화가 너무 지루해서 하나도 안 재미있어요 : 별점: 5점(99.21%))
배우 연기가 별로네요 : 별점: 3점(98.54%))
